# Edge IIoT - Binary Classification


In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
import torch

print(torch.__version__) # X.XX.X+cuXXX / If X.XX.X+cpu it won't work
print(torch.cuda.is_available()) # False
print(torch.version.cuda) # None or mismatched version
print(torch.cuda.device_count()) # 0

2.5.1+cu121
True
12.1
1


In [3]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.config import DATASETS, SEED
dataset_name = "edge_iiot"
config = DATASETS[dataset_name]

print("\n--- Dataset Information ---")
print(f"Name: {dataset_name.upper()}")
print(f"Path: {config['processed_path']}\n")

filename = "ML-EdgeIIoT-dataset"

csv_path = config['processed_path'] / f"{filename}.pkl"
print(f"CSV Path: {csv_path}\n")

print("Loading dataset... (This may take a while)")
df = pd.read_pickle(csv_path)

print(f"\nDataset loaded with shape: {df.shape}")


--- Dataset Information ---
Name: EDGE_IIOT
Path: /home/uo294319/ML-NIDS-IIoT/data/edge_iiot/processed

CSV Path: /home/uo294319/ML-NIDS-IIoT/data/edge_iiot/processed/ML-EdgeIIoT-dataset.pkl

Loading dataset... (This may take a while)

Dataset loaded with shape: (152590, 62)


## 1. Undersampling and Balancing

In [4]:
target     = 'Attack_label'
target_str = 'Attack_type'

In [5]:
y_str = df[target_str]
df_no_label = df.drop(columns=[target_str])

In [6]:
MAX_PRESENCE = 0.02

counts = y_str.value_counts()
total_rows = len(y_str)

print(counts)
print("\nTOTAL: ", total_rows)

Attack_type
Normal                   24301
DDoS_UDP                 14498
DDoS_ICMP                13307
Ransomware               10925
DDoS_HTTP                10561
SQL_injection            10311
Uploading                10269
Backdoor                 10195
Vulnerability_scanner    10075
Port_Scanning            10071
XSS                      10051
Password                  9989
DDoS_TCP                  6011
MITM                      1028
Fingerprinting             998
Name: count, dtype: int64

TOTAL:  152590


In [7]:
print(f"{'Attack Type':<25} | {'Old Count':<15} | {'New Count':<15}\n" + "-"*55)

sampling_strategy = {}
for attack_type, count in counts.items():
    current_presence = count / total_rows

    if current_presence > MAX_PRESENCE:
        new_count = int(total_rows * MAX_PRESENCE)
    else:
        new_count = count

    sampling_strategy[attack_type] = new_count
    print(f"{attack_type:<25} | {count:<15} | {new_count:<15}")


print("-"*55 + f"\n{'TOTAL':<25} | {counts.sum():<15} | {sum(sampling_strategy.values()):<15}")


Attack Type               | Old Count       | New Count      
-------------------------------------------------------
Normal                    | 24301           | 3051           
DDoS_UDP                  | 14498           | 3051           
DDoS_ICMP                 | 13307           | 3051           
Ransomware                | 10925           | 3051           
DDoS_HTTP                 | 10561           | 3051           
SQL_injection             | 10311           | 3051           
Uploading                 | 10269           | 3051           
Backdoor                  | 10195           | 3051           
Vulnerability_scanner     | 10075           | 3051           
Port_Scanning             | 10071           | 3051           
XSS                       | 10051           | 3051           
Password                  | 9989            | 3051           
DDoS_TCP                  | 6011            | 3051           
MITM                      | 1028            | 1028           
Fingerprinting

In [8]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=SEED)
df_no_label, y_str = rus.fit_resample(df_no_label, y_str)

print(df_no_label.shape)

(41689, 61)


In [9]:
# X/y split
X     = df_no_label.drop(columns=[target])
y     = df_no_label[target]

## 1. Pre-processing

In [10]:
# Select only numeric

print("--- Non-numeric cols to drop ---\n\n", X.select_dtypes(include=['str', 'object', 'category']).columns)

X = X.select_dtypes(include=['number'])

print("\n\nRemaining categorical cols:", len(X.select_dtypes(include=['str', 'object', 'category']).columns))

--- Non-numeric cols to drop ---

 Index(['http.file_data', 'http.referer', 'http.request.path',
       'http.request.uri.query', 'http.request.version', 'mqtt.msg', 'proto'],
      dtype='str')


Remaining categorical cols: 0


In [11]:
print(f"NaN values in target variable: {y.isna().sum()}")
print(f"NaN values in features: {X.isna().sum().sum()}")

NaN values in target variable: 0
NaN values in features: 0


In [12]:
X_train, X_test, y_train, y_test, y_str_train, y_str_test = train_test_split(
    X, y, y_str, test_size=0.2, random_state=SEED, stratify=y_str
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (33351, 53)
X_test shape: (8338, 53)


In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_results = {}

## 2. LazyPredict
[Docs](https://pypi.org/project/lazypredict/)

In [14]:
from lazypredict.Supervised import LazyClassifier

# With categorical encoding, timeout, cross-validation, and GPU
clf = LazyClassifier(
    verbose=1,                          # Show progress
    ignore_warnings=True,               # Suppress warnings
    custom_metric=None,                 # Use default metrics
    predictions=False,                  # Don't Return predictions
    classifiers='all',                  # Use all available classifiers
    timeout=60,                         # Max time per model in seconds
    cv=5,                               # Cross-validation folds (optional)
)

models, _ = clf.fit(X_train, X_test, y_train, y_test)
print("\n--- Models Evaluated ---")

  0%|          | 0/32 [00:00<?, ?it/s]

/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
/home/uo294319/ML-NIDS-IIoT/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1


--- Models Evaluated ---


In [15]:
display(models)

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Accuracy CV Mean,Accuracy CV Std,Balanced Accuracy CV Mean,Balanced Accuracy CV Std,ROC AUC CV Mean,ROC AUC CV Std,F1 Score CV Mean,F1 Score CV Std,Precision CV Mean,Precision CV Std,Recall CV Mean,Recall CV Std,Time Taken
Model,,,,,,,,,,,,,,,,,,,
LGBMClassifier,0.999041,0.995708,0.999897,0.999040,0.999040,0.999041,0.998921,0.000580,0.995080,0.002795,0.999775,0.000186,0.998919,0.000581,0.998919,0.000582,0.998921,0.000580,350.628665
XGBClassifier,0.999280,0.995082,0.999698,0.999279,0.999281,0.999280,0.999130,0.000334,0.995193,0.001916,0.999894,0.000090,0.999129,0.000335,0.999130,0.000335,0.999130,0.000334,2.229395
ExtraTreesClassifier,0.999160,0.995017,0.999158,0.999159,0.999160,0.999160,0.999190,0.000120,0.996733,0.000600,0.999781,0.000417,0.999190,0.000120,0.999190,0.000119,0.999190,0.000120,1.182661
RandomForestClassifier,0.999041,0.994953,0.999551,0.999039,0.999039,0.999041,0.999160,0.000336,0.995775,0.001346,0.999858,0.000257,0.999160,0.000337,0.999160,0.000337,0.999160,0.000336,2.979835
ExtraTreeClassifier,0.998321,0.993809,0.993809,0.998321,0.998321,0.998321,0.997841,0.000398,0.992234,0.001645,0.992234,0.001645,0.997842,0.000398,0.997842,0.000398,0.997841,0.000398,1.296776
BaggingClassifier,0.997961,0.992860,0.998178,0.997962,0.997963,0.997961,0.998291,0.000611,0.993985,0.002945,0.997592,0.002040,0.998291,0.000612,0.998292,0.000613,0.998291,0.000611,2.207799
DecisionTreeClassifier,0.997841,0.991286,0.991286,0.997840,0.997838,0.997841,0.997841,0.000688,0.991101,0.003139,0.991101,0.003139,0.997839,0.000686,0.997842,0.000683,0.997841,0.000688,1.496979
AdaBoostClassifier,0.990405,0.939711,0.997193,0.990151,0.990378,0.990405,0.989565,0.001431,0.936826,0.010898,0.996963,0.000610,0.989284,0.001519,0.989519,0.001426,0.989565,0.001431,2.845878
KNeighborsClassifier,0.988486,0.927351,0.978424,0.988112,0.988450,0.988486,0.985668,0.001470,0.912277,0.006500,0.971678,0.003612,0.985121,0.001534,0.985535,0.001583,0.985668,0.001470,1.583096


In [16]:
display(models.sort_values(by='F1 Score CV Mean', ascending=False).head(3))

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Accuracy CV Mean,Accuracy CV Std,Balanced Accuracy CV Mean,Balanced Accuracy CV Std,ROC AUC CV Mean,ROC AUC CV Std,F1 Score CV Mean,F1 Score CV Std,Precision CV Mean,Precision CV Std,Recall CV Mean,Recall CV Std,Time Taken
Model,,,,,,,,,,,,,,,,,,,
ExtraTreesClassifier,0.999160,0.995017,0.999158,0.999159,0.999160,0.999160,0.99919,0.000120,0.996733,0.000600,0.999781,0.000417,0.999190,0.000120,0.99919,0.000119,0.99919,0.000120,1.182661
RandomForestClassifier,0.999041,0.994953,0.999551,0.999039,0.999039,0.999041,0.99916,0.000336,0.995775,0.001346,0.999858,0.000257,0.999160,0.000337,0.99916,0.000337,0.99916,0.000336,2.979835
XGBClassifier,0.999280,0.995082,0.999698,0.999279,0.999281,0.999280,0.99913,0.000334,0.995193,0.001916,0.999894,0.000090,0.999129,0.000335,0.99913,0.000335,0.99913,0.000334,2.229395
